# 05 — Append-Only Filtering (FilteringPress)

This notebook demonstrates the **append-only filtering** compression
strategy: during generation, each new token is scored and either kept
or skipped before entering the cache. The cache only grows, never
shrinks — compression comes from skipping low-scoring tokens.

This mirrors kvpress's `FilteringPress`, which wraps a scoring press
(like `KeyDiffPress`) and applies a keep/skip threshold at each decode
step. Decisions are made **per head** independently — each head scores
all tokens and decides whether the new one survives. When a head
rejects a token, that position becomes padding for that head only,
creating ragged per-head lengths. The token is removed from the cache
only when **all heads** reject it.

We run three experiments:
1. **Filtering decisions** — simulate decode steps with per-head
   keep/skip decisions and ragged length tracking
2. **Cross-validation** — verify our decisions match kvpress's
   `FilteringPress` on the same key sequence
3. **Full pipeline on paged cache** — same filtering but with keys
   living in the Flash Attention paged cache

## Imports and Setup

In [ ]:
import sys

import torch
import torch.nn.functional as F
from vllm import _custom_ops as ops

torch.manual_seed(42)
torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
sys.path.insert(0, "/opt/app-root/src/kvpress-fork")
from kvpress import KeyDiffPress, FilteringPress

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
PREFILL_LEN = 0
NUM_DECODE_STEPS = 16384
COMPRESSION_RATIO = 0.5
DEVICE = "cuda"

## Primitives

Cache operations from notebook 02, scoring from notebook 03, plus
helpers for FilteringPress's per-head ragged length management:
`build_valid_mask`, `accept_last`, `fill_padding`, and `shrink`.

In [ ]:
def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


def build_slot_mapping_for_positions(block_table, positions, block_size):
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size
    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]
    return keys, values


print("Cache functions defined")

In [ ]:
def keydiff_score(keys):
    """Score keys using KeyDiff's key-similarity metric.

    keys: [seq_len, head_dim]
    Returns: [seq_len]
    """
    normalized = F.normalize(keys, p=2, dim=-1)
    anchor = normalized.mean(dim=0, keepdim=True)
    return -F.cosine_similarity(keys, anchor, dim=-1)


def score_per_head(keys, valid_mask):
    """Score all heads by looping, passing only valid positions to the scorer.

    keys: [seq_len, num_kv_heads, head_dim]
    valid_mask: [num_kv_heads, seq_len]
    Returns: [num_kv_heads, seq_len] with -inf for invalid positions.
    """
    num_heads = keys.shape[1]
    seq_len = keys.shape[0]
    scores = torch.full((num_heads, seq_len), float("-inf"), device=keys.device, dtype=keys.dtype)
    for h in range(num_heads):
        valid_pos = valid_mask[h].nonzero(as_tuple=True)[0]
        head_scores = keydiff_score(keys[valid_pos, h, :])
        scores[h, valid_pos] = head_scores
    return scores


def build_valid_mask(lengths, seq_len, device):
    """Per-head valid mask from ragged lengths, with the last position
    always valid (the new token being evaluated).

    lengths: [num_kv_heads]
    Returns: [num_kv_heads, seq_len]
    """
    positions = torch.arange(seq_len, device=device).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)
    mask[:, -1] = True
    return mask


def accept_last(keys, lengths, accepted):
    """Move the last position into each accepting head's first padding
    slot, keeping valid data packed as a prefix.

    keys: [seq_len, num_kv_heads, head_dim] — modified in place
    lengths: [num_kv_heads] — modified in place
    accepted: [num_kv_heads] bool
    """
    last_pos = keys.shape[0] - 1
    for h in range(keys.shape[1]):
        if not accepted[h]:
            continue
        gap = lengths[h].item()
        if gap < last_pos:
            keys[gap, h] = keys[last_pos, h]
            keys[last_pos, h] = 0.0
        lengths[h] += 1


def fill_padding(keys, lengths):
    """Zero out padding positions per head."""
    for h in range(keys.shape[1]):
        L = lengths[h].item()
        if L < keys.shape[0]:
            keys[L:, h] = 0.0


def shrink(keys, lengths):
    """Remove trailing positions that are padding for all heads."""
    while keys.shape[0] > 0 and lengths.max().item() < keys.shape[0]:
        keys = keys[:-1]
    return keys


print("Scoring and ragged-length functions defined")

## Experiment 1 — Filtering Decisions

Simulate a sequence of decode steps. At each step:
1. Append the new token's key to the cache
2. Build a per-head valid mask (prefix per head + last position)
3. Score all keys with KeyDiff, masking invalid positions to `-inf`
4. Compute per-head rejection: a head rejects when the new token's
   score is strictly below the top-k threshold
5. Skip the token only when **all heads** reject it; otherwise accept
   per-head (rejecting heads get padding at that position)

In [ ]:
all_keys = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

print(f"Prefill tokens:     {PREFILL_LEN}")
print(f"Decode steps:       {NUM_DECODE_STEPS}")
print(f"Compression ratio:  {COMPRESSION_RATIO}")
print(f"Total keys:         {all_keys.shape[0]}")

In [ ]:
cached_keys = all_keys[:PREFILL_LEN].clone()
lengths = torch.full((NUM_KV_HEADS,), PREFILL_LEN, dtype=torch.long, device=DEVICE)
decisions = []

for step in range(NUM_DECODE_STEPS):
    new_key = all_keys[PREFILL_LEN + step]
    total_tokens_seen = PREFILL_LEN + step + 1
    cache_len_before = cached_keys.shape[0]

    keys_with_new = torch.cat([cached_keys, new_key.unsqueeze(0)], dim=0)
    seq_len = keys_with_new.shape[0]

    valid_mask = build_valid_mask(lengths, seq_len, DEVICE)

    scores = score_per_head(keys_with_new, valid_mask)

    n_kept = round(total_tokens_seen * (1 - COMPRESSION_RATIO))
    n_kept = max(1, min(n_kept, seq_len))
    threshold = scores.topk(n_kept, dim=-1, sorted=True).values[:, -1]
    rejected = scores[:, -1] < threshold

    keep = not rejected.all().item()

    if keep:
        accepted = ~rejected
        accept_last(keys_with_new, lengths, accepted)
        fill_padding(keys_with_new, lengths)
        keys_with_new = shrink(keys_with_new, lengths)
        cached_keys = keys_with_new

    decisions.append({
        "step": step,
        "total_seen": total_tokens_seen,
        "cache_len_before": cache_len_before,
        "rejected_per_head": rejected.clone(),
        "keep": keep,
        "lengths_after": lengths.clone(),
    })

per_head_accepted = torch.zeros(NUM_KV_HEADS, dtype=torch.long)
for d in decisions:
    per_head_accepted += (~d["rejected_per_head"]).long().cpu()

print(f"Results:")
print(f"  Final cache size: {cached_keys.shape[0]} tokens")
print(f"  Per-head lengths: {lengths.tolist()}")
print(f"  Per-head decode tokens accepted:")
for h in range(NUM_KV_HEADS):
    kept = per_head_accepted[h].item()
    print(f"    Head {h}: {kept}/{NUM_DECODE_STEPS} ({kept/NUM_DECODE_STEPS:.1%})")
avg_rate = per_head_accepted.float().mean().item() / NUM_DECODE_STEPS
print(f"  Average per-head keep rate: {avg_rate:.1%}")
print()

log_interval = max(1, NUM_DECODE_STEPS // 20)
for d in decisions:
    if d["step"] % log_interval != 0 and d["step"] != NUM_DECODE_STEPS - 1:
        continue
    accepted = (~d['rejected_per_head']).long()
    n_accepted = accepted.sum().item()
    heads_str = ''.join('K' if a else '.' for a in accepted.tolist())
    print(
        f"  step {d['step']:5d}  seen={d['total_seen']:5d}  "
        f"cache={d['cache_len_before']:5d}  "
        f"heads=[{heads_str}]  accepted={n_accepted}/{NUM_KV_HEADS}  "
        f"lengths={d['lengths_after'].tolist()}"
    )

## Experiment 1c — Divergence Diagnostic

Run the coupled multi-head loop and independent single-head loops
side by side, comparing per-head decisions at every step. Print
the first divergence for each head with full details: n_kept,
valid_count, threshold, new token score, and decision.

In [ ]:
ind_caches = [all_keys[:PREFILL_LEN, h:h+1, :].clone() for h in range(NUM_KV_HEADS)]
ind_lens = [torch.full((1,), PREFILL_LEN, dtype=torch.long, device=DEVICE) for _ in range(NUM_KV_HEADS)]

cpl_cache = all_keys[:PREFILL_LEN].clone()
cpl_lens = torch.full((NUM_KV_HEADS,), PREFILL_LEN, dtype=torch.long, device=DEVICE)

first_divergence = [None] * NUM_KV_HEADS
total_divergences = [0] * NUM_KV_HEADS

for step in range(NUM_DECODE_STEPS):
    total = PREFILL_LEN + step + 1
    nk_raw = round(total * (1 - COMPRESSION_RATIO))

    # --- coupled (multi-head) ---
    new_key = all_keys[PREFILL_LEN + step]
    cpl_kw = torch.cat([cpl_cache, new_key.unsqueeze(0)], dim=0)
    cpl_sl = cpl_kw.shape[0]
    cpl_nk = max(1, min(nk_raw, cpl_sl))
    cpl_vm = build_valid_mask(cpl_lens, cpl_sl, DEVICE)
    cpl_sc = score_per_head(cpl_kw, cpl_vm)
    cpl_thr = cpl_sc.topk(cpl_nk, dim=-1, sorted=True).values[:, -1]
    cpl_rej = cpl_sc[:, -1] < cpl_thr
    cpl_keep = not cpl_rej.all().item()

    if cpl_keep:
        cpl_acc = ~cpl_rej
        accept_last(cpl_kw, cpl_lens, cpl_acc)
        fill_padding(cpl_kw, cpl_lens)
        cpl_kw = shrink(cpl_kw, cpl_lens)
        cpl_cache = cpl_kw

    # --- independent (per head) ---
    for h in range(NUM_KV_HEADS):
        hk = all_keys[PREFILL_LEN + step, h:h+1, :].unsqueeze(0)
        ind_kw = torch.cat([ind_caches[h], hk], dim=0)
        ind_sl = ind_kw.shape[0]
        ind_nk = max(1, min(nk_raw, ind_sl))
        ind_vm = build_valid_mask(ind_lens[h], ind_sl, DEVICE)
        ind_sc = score_per_head(ind_kw, ind_vm)
        ind_thr = ind_sc.topk(ind_nk, dim=-1, sorted=True).values[:, -1]
        ind_rej = ind_sc[:, -1] < ind_thr
        ind_keep = not ind_rej.all().item()

        if ind_keep:
            ind_acc = ~ind_rej
            accept_last(ind_kw, ind_lens[h], ind_acc)
            fill_padding(ind_kw, ind_lens[h])
            ind_kw = shrink(ind_kw, ind_lens[h])
            ind_caches[h] = ind_kw

        # --- compare ---
        cpl_h_rej = cpl_rej[h].item()
        ind_h_rej = ind_rej[0].item()

        if cpl_h_rej != ind_h_rej:
            total_divergences[h] += 1
            if first_divergence[h] is None:
                cpl_valid = cpl_vm[h].sum().item()
                ind_valid = ind_vm[0].sum().item()
                first_divergence[h] = {
                    "step": step,
                    "n_kept_coupled": cpl_nk,
                    "n_kept_indep": ind_nk,
                    "valid_count_coupled": cpl_valid,
                    "valid_count_indep": ind_valid,
                    "seq_len_coupled": cpl_sl,
                    "seq_len_indep": ind_sl,
                    "threshold_coupled": cpl_thr[h].item(),
                    "threshold_indep": ind_thr[0].item(),
                    "new_score_coupled": cpl_sc[h, -1].item(),
                    "new_score_indep": ind_sc[0, -1].item(),
                    "anchor_diff": (
                        cpl_sc[h, cpl_vm[h]].float() - ind_sc[0, ind_vm[0]].float()
                    ).abs().max().item(),
                    "coupled_rejected": cpl_h_rej,
                    "indep_rejected": ind_h_rej,
                }

print("Divergence diagnostic (coupled vs independent):\n")
for h in range(NUM_KV_HEADS):
    if first_divergence[h] is None:
        print(f"  Head {h}: no divergence in {NUM_DECODE_STEPS} steps")
    else:
        d = first_divergence[h]
        print(f"  Head {h}: first divergence at step {d['step']} "
              f"({total_divergences[h]} total divergences)")
        print(f"    n_kept:       coupled={d['n_kept_coupled']}  indep={d['n_kept_indep']}")
        print(f"    valid_count:  coupled={d['valid_count_coupled']}  indep={d['valid_count_indep']}")
        print(f"    seq_len:      coupled={d['seq_len_coupled']}  indep={d['seq_len_indep']}")
        print(f"    threshold:    coupled={d['threshold_coupled']:.6f}  indep={d['threshold_indep']:.6f}")
        print(f"    new_score:    coupled={d['new_score_coupled']:.6f}  indep={d['new_score_indep']:.6f}")
        print(f"    max |score diff| across valid positions: {d['anchor_diff']:.6e}")
        print(f"    decision:     coupled={'REJECT' if d['coupled_rejected'] else 'KEEP'}"
              f"  indep={'REJECT' if d['indep_rejected'] else 'KEEP'}")
        print()

## Experiment 2 — Cross-Validation Against kvpress

Run the same key sequence through kvpress's `FilteringPress` and verify
that the per-head lengths match our standalone implementation after each
decode step.

This requires a mock `nn.Module` with a `layer_idx` attribute, and
`position_ids` in kwargs — the only external dependencies of
`FilteringPress.compress()`.

In [ ]:
from types import SimpleNamespace

mock_module = SimpleNamespace(layer_idx=0, head_dim=HEAD_SIZE)

fp = FilteringPress(
    base_press=KeyDiffPress(),
    target_compression_ratio=COMPRESSION_RATIO,
)

kvpress_keys = all_keys[:PREFILL_LEN].permute(1, 0, 2).unsqueeze(0).clone()
kvpress_values = torch.zeros_like(kvpress_keys)

length_mismatches = 0
fp.reset()

for step in range(NUM_DECODE_STEPS):
    new_key = all_keys[PREFILL_LEN + step]
    total_tokens_seen = PREFILL_LEN + step + 1

    new_key_kvpress = new_key.unsqueeze(0).unsqueeze(0).permute(0, 2, 1, 3)
    keys_in = torch.cat([kvpress_keys, new_key_kvpress], dim=2)
    values_in = torch.cat(
        [kvpress_values, torch.zeros_like(new_key_kvpress)], dim=2,
    )

    position_ids = torch.arange(total_tokens_seen, device=DEVICE).unsqueeze(0)

    keys_out, values_out = fp.compress(
        module=mock_module,
        hidden_states=None,
        keys=keys_in,
        values=values_in,
        attentions=None,
        kwargs={"position_ids": position_ids},
    )

    if 0 in fp._lengths:
        kvpress_lengths = fp._lengths[0][0]
    else:
        kvpress_lengths = torch.full(
            (NUM_KV_HEADS,), PREFILL_LEN, dtype=torch.long, device=DEVICE,
        )

    our_lengths = decisions[step]["lengths_after"]

    if not torch.equal(our_lengths, kvpress_lengths):
        length_mismatches += 1
        print(
            f"  LENGTH MISMATCH step {step}: "
            f"ours={our_lengths.tolist()}, kvpress={kvpress_lengths.tolist()}"
        )

    kvpress_keys = keys_out
    kvpress_values = values_out

# Compare per-head valid key data at the final state
key_mismatches = 0
for h in range(NUM_KV_HEADS):
    our_len = lengths[h].item()
    kv_len = kvpress_lengths[h].item()

    if our_len != kv_len:
        key_mismatches += 1
        print(f"  KEY DATA head {h}: cannot compare — lengths differ ({our_len} vs {kv_len})")
        continue

    our_valid = cached_keys[:our_len, h, :]
    kv_valid = kvpress_keys[0, h, :kv_len, :]

    if not torch.equal(our_valid, kv_valid):
        key_mismatches += 1
        diff = (our_valid.float() - kv_valid.float()).abs().max(dim=-1).values
        mismatch_pos = (diff > 0).nonzero(as_tuple=True)[0]
        print(
            f"  KEY DATA head {h}: {len(mismatch_pos)}/{our_len} positions differ, "
            f"max_diff={diff.max():.6e}"
        )

if length_mismatches == 0 and key_mismatches == 0:
    print(
        f"All {NUM_DECODE_STEPS} filtering decisions match kvpress "
        f"FilteringPress (per-head lengths and key data identical)"
    )
else:
    if length_mismatches > 0:
        print(f"\n{length_mismatches}/{NUM_DECODE_STEPS} length mismatches")
    if key_mismatches > 0:
        print(f"{key_mismatches}/{NUM_KV_HEADS} heads have key data mismatches")

## Experiment 3 — Full Pipeline on Paged Cache

End-to-end: keys live in the Flash Attention paged cache. At each decode
step, write the new token to the cache, gather all keys, score with the
valid mask, and apply per-head filtering with gap-filling — all operating
directly on paged cache slots.

In [ ]:
def accept_last_paged(key_cache, value_cache, block_table, block_size,
                      new_pos, lengths, accepted, device):
    """Move the new token from new_pos to each accepting head's first
    gap in the paged cache."""
    new_slot = build_slot_mapping_for_positions(
        block_table, torch.tensor([new_pos], device=device), block_size,
    )
    new_block = (new_slot // block_size).item()
    new_offset = (new_slot % block_size).item()

    for h in range(len(lengths)):
        if not accepted[h]:
            continue
        gap = lengths[h].item()
        if gap < new_pos:
            gap_slot = build_slot_mapping_for_positions(
                block_table, torch.tensor([gap], device=device), block_size,
            )
            gap_block = (gap_slot // block_size).item()
            gap_offset = (gap_slot % block_size).item()
            key_cache[gap_block, gap_offset, h] = key_cache[new_block, new_offset, h]
            value_cache[gap_block, gap_offset, h] = value_cache[new_block, new_offset, h]
            key_cache[new_block, new_offset, h] = 0.0
            value_cache[new_block, new_offset, h] = 0.0
        lengths[h] += 1


def fill_padding_paged(key_cache, value_cache, block_table, block_size,
                       max_pos, lengths, device):
    """Zero out padding positions per head in the paged cache."""
    for h in range(len(lengths)):
        L = lengths[h].item()
        if L >= max_pos:
            continue
        padding_positions = torch.arange(L, max_pos, dtype=torch.long, device=device)
        padding_slots = build_slot_mapping_for_positions(
            block_table, padding_positions, block_size,
        )
        blocks = padding_slots // block_size
        offsets = padding_slots % block_size
        key_cache[blocks, offsets, h] = 0.0
        value_cache[blocks, offsets, h] = 0.0


all_values = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

total_len = PREFILL_LEN + NUM_DECODE_STEPS
num_blocks = (total_len + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

num_seq_blocks = (total_len + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)
k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

if PREFILL_LEN > 0:
    prefill_positions = torch.arange(PREFILL_LEN, dtype=torch.long, device=DEVICE)
    prefill_slots = build_slot_mapping_for_positions(
        block_table, prefill_positions, BLOCK_SIZE,
    )
    ops.reshape_and_cache_flash(
        all_keys[:PREFILL_LEN], all_values[:PREFILL_LEN],
        key_cache, value_cache,
        prefill_slots, "auto", k_scale, v_scale,
    )

print("Paged-cache filtering helpers defined")
print(f"Prefilled {PREFILL_LEN} tokens into paged cache")

In [ ]:
max_cache_pos = PREFILL_LEN
paged_lengths = torch.full((NUM_KV_HEADS,), PREFILL_LEN, dtype=torch.long, device=DEVICE)
paged_decisions = []

for step in range(NUM_DECODE_STEPS):
    total_tokens_seen = PREFILL_LEN + step + 1
    new_key = all_keys[PREFILL_LEN + step].unsqueeze(0)
    new_value = all_values[PREFILL_LEN + step].unsqueeze(0)

    new_position = torch.tensor([max_cache_pos], dtype=torch.long, device=DEVICE)
    new_slot = build_slot_mapping_for_positions(
        block_table, new_position, BLOCK_SIZE,
    )
    ops.reshape_and_cache_flash(
        new_key, new_value,
        key_cache, value_cache,
        new_slot, "auto", k_scale, v_scale,
    )

    all_positions = torch.arange(max_cache_pos + 1, dtype=torch.long, device=DEVICE)
    all_slots = build_slot_mapping_for_positions(
        block_table, all_positions, BLOCK_SIZE,
    )
    cached_keys_paged, _ = gather_from_paged_cache(
        key_cache, value_cache, all_slots, BLOCK_SIZE,
    )

    seq_len = cached_keys_paged.shape[0]
    valid_mask = build_valid_mask(paged_lengths, seq_len, DEVICE)
    scores = score_per_head(cached_keys_paged, valid_mask)

    n_kept = round(total_tokens_seen * (1 - COMPRESSION_RATIO))
    n_kept = max(1, min(n_kept, seq_len))
    threshold = scores.topk(n_kept, dim=-1, sorted=True).values[:, -1]
    rejected = scores[:, -1] < threshold

    keep = not rejected.all().item()
    paged_decisions.append({"keep": keep, "rejected_per_head": rejected.clone()})

    if not keep:
        block_idx = (new_slot // BLOCK_SIZE).item()
        offset = (new_slot % BLOCK_SIZE).item()
        key_cache[block_idx, offset] = 0.0
        value_cache[block_idx, offset] = 0.0
        continue

    accepted = ~rejected
    accept_last_paged(
        key_cache, value_cache, block_table, BLOCK_SIZE,
        max_cache_pos, paged_lengths, accepted, DEVICE,
    )
    fill_padding_paged(
        key_cache, value_cache, block_table, BLOCK_SIZE,
        max_cache_pos + 1, paged_lengths, DEVICE,
    )

    if paged_lengths.max().item() >= max_cache_pos + 1:
        max_cache_pos += 1

paged_per_head_accepted = torch.zeros(NUM_KV_HEADS, dtype=torch.long)
for d in paged_decisions:
    paged_per_head_accepted += (~d["rejected_per_head"]).long().cpu()

print(f"Final max cache position: {max_cache_pos}")
print(f"Per-head lengths: {paged_lengths.tolist()}")
print(f"Per-head decode tokens accepted:")
for h in range(NUM_KV_HEADS):
    kept = paged_per_head_accepted[h].item()
    print(f"  Head {h}: {kept}/{NUM_DECODE_STEPS} ({kept/NUM_DECODE_STEPS:.1%})")
print()

dense_keep = [d["keep"] for d in decisions]
paged_keep = [d["keep"] for d in paged_decisions]
mismatches = sum(1 for p, d in zip(paged_keep, dense_keep) if p != d)

if mismatches == 0:
    print(
        f"All {NUM_DECODE_STEPS} decisions match between "
        f"paged-cache and dense-tensor pipelines"
    )
else:
    print(f"{mismatches}/{NUM_DECODE_STEPS} decisions differ")
    for i, (p, d) in enumerate(zip(paged_keep, dense_keep)):
        if p != d:
            print(f"  step {i}: paged={p}, dense={d}")

## Notes and Next Steps

**FilteringPress emulation validated.** The standalone per-head filtering
logic makes the same keep/skip decisions as kvpress's `FilteringPress`,
including ragged per-head lengths, accept-last gap-filling, and padding
management. These decisions are identical whether operating on dense
tensors or on gathered paged cache data.

**What this enables:** With scoring and per-head filtering validated on
the Flash Attention cache layout, the next step is integrating this into
vLLM's `Attention.forward()`. The integration will:
1. Gather cached keys after `do_kv_cache_update`
2. Score with `keydiff_score`
3. Apply per-head filtering with ragged length tracking
4. Manage gap-filling and padding in the paged cache
5. Track logical positions for RoPE separately from cache positions